# Silver — conform the three coverage series

Bronze made no decisions. Silver makes them all, in one place, visibly.

Three publication series arrive in three shapes, two coverage scales, two region systems and two
age vocabularies. They leave as one long table where a row means the same thing whatever it came
from. Reference date and publication date stay in separate columns.

Reads `data/bronze/tables/`. Requires `01_bronze.ipynb` to have run.

## 1. Paths and inputs

In [3]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

BASE_DIR   = Path.home() / "projects" / "nhs-cervical-screening-pipeline"
BRONZE_DIR = BASE_DIR / "data" / "bronze"
TABLES_DIR = BRONZE_DIR / "tables"
SILVER_DIR = BASE_DIR / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(BRONZE_DIR / "manifest.parquet")

def bronze(name):
    return pd.read_parquet(TABLES_DIR / f"{name}.parquet")

# The conformed schema. Every conform function returns exactly these columns, in this order.
CANON = ["series", "source_id", "publication_date", "publication_date_basis",
         "period_end", "geography_level", "geography_system", "org_code", "org_name",
         "age_band", "age_band_source", "eligible", "screened",
         "coverage_pct_published", "coverage_pct_computed", "screens_to_target",
         "quarter", "comparable", "comparability_note"]

def to_num(s):
    return pd.to_numeric(s.replace("", np.nan), errors="coerce")

NUMERIC = {"eligible", "screened", "coverage_pct_published",
           "coverage_pct_computed", "screens_to_target"}

def finish(df):
    """Fill any canonical column a source does not supply, with a settled dtype.

    Typing the gaps here rather than leaving bare NA keeps pandas from having to guess
    dtypes when these frames are concatenated."""
    for c in CANON:
        if c not in df.columns:
            df[c] = np.nan if c in NUMERIC else pd.NA
    for c in NUMERIC:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    for c in set(CANON) - NUMERIC - {"comparable"}:
        df[c] = df[c].astype("object")
    return df[CANON]

print(f"{len(manifest)} sources in the Bronze manifest, "
      f"{len(list(TABLES_DIR.glob('*.parquet')))} tables landed")

13 sources in the Bronze manifest, 14 tables landed


## 2. Conform the annual publication

`phof.csv` is wide: three age bands across nine columns. It becomes long.

The period end is derived per row from `CollectionYearRange`, not taken from the manifest — the
manifest records the reference date of the publication, but this file carries eleven years and each
year has its own. The two are cross-checked later.

This is the only source that publishes a 25-64 figure. Everywhere else it has to be derived.

In [5]:
# band -> (eligible column, screened column, published coverage column)
PHOF_BANDS = {
    "25-64": ("Eligible_PHOF_25_64", "Screened_PHOF_42_66mth", "Coverage_PHOF_25_64"),
    "25-49": ("Eligible_PHOF_25_49", "Screened_PHOF_42mth",    "Coverage_PHOF_25_49"),
    "50-64": ("Eligible_PHOF_50_64", "Screened_PHOF_66mth",    "Coverage_PHOF_50_64"),
}

ORG_TYPE_TO_LEVEL = {"National": "National", "Region": "Region", "LA": "Local Authority"}

def conform_annual_phof():
    raw = bronze("annual_phof")
    out = []
    for band, (el, sc, cov) in PHOF_BANDS.items():
        d = pd.DataFrame({
            "series": raw["_series"],
            "source_id": raw["_source_id"],
            "publication_date": raw["_publication_date"],
            "publication_date_basis": raw["_publication_date_basis"],
            # "2023-24" describes the position at 31 March of the closing year
            "period_end": raw.CollectionYearRange.str[:4].astype(int).add(1)
                             .astype(str) + "-03-31",
            "geography_level": raw.Org_Type.map(ORG_TYPE_TO_LEVEL),
            # the annual file uses ONS statistical regions; the GOV.UK files do not
            "geography_system": np.where(raw.Org_Type == "Region", "ONS", pd.NA),
            "org_code": raw.Org_ONS_Code,
            "org_name": raw.Org_Name,
            "age_band": band,
            "age_band_source": "published",
            "eligible": to_num(raw[el]),
            "screened": to_num(raw[sc]),
            "coverage_pct_published": to_num(raw[cov]),
            "comparable": True,
            "comparability_note": "",
        })
        out.append(d)
    return finish(pd.concat(out, ignore_index=True))

annual = conform_annual_phof()
print(f"annual: {len(annual):,} rows, {annual.period_end.nunique()} periods, "
      f"{annual.geography_level.nunique()} geography levels")
annual.head(3)

annual: 990 rows, 11 periods, 3 geography levels


,series,source_id,publication_date,publication_date_basis,period_end,geography_level,geography_system,org_code,org_name,age_band,age_band_source,eligible,screened,coverage_pct_published,coverage_pct_computed,screens_to_target,quarter,comparable,comparability_note
0,annual,annual_phof,2024-11-28,stated on publication page,2014-03-31,National,<NA>,E92000001,England,25-64,published,13954489.0,10348915.0,74.161906,NaN,NaN,<NA>,True,
1,annual,annual_phof,2024-11-28,stated on publication page,2015-03-31,National,<NA>,E92000001,England,25-64,published,14165803.0,10405033.0,73.451770,NaN,NaN,<NA>,True,
2,annual,annual_phof,2024-11-28,stated on publication page,2016-03-31,National,<NA>,E92000001,England,25-64,published,14398174.0,10464175.0,72.677098,NaN,NaN,<NA>,True,


## 3. Conform the quarterly management information

Already long by age band, so the work here is renaming and flagging.

Two flags matter. The Sub-ICB file's final quarter, 2024-25 Q1, was extracted from NHAIS at system
closure without the usual two-month lag; the publisher states it runs low. It is marked not
comparable so no chart downstream can plot it as a fall. And the two files name the same column
differently — `No. screens to reach 80%` against `No. screens to reach target` — which is exactly
the kind of drift a conforming layer exists to absorb.

In [7]:
NOT_COMPARABLE_QUARTER = "2024-25 Q1"
NHAIS_NOTE = ("Extracted from NHAIS at closure on 24 June 2024 without the usual two-month lag; "
              "the publisher states coverage runs lower than expected. Not comparable with "
              "earlier quarters.")

def conform_mi(source_id, level):
    raw = bronze(source_id)
    screens_col = next((c for c in raw.columns if c.startswith("No. screens to reach")), None)

    d = pd.DataFrame({
        "series": raw["_series"],
        "source_id": raw["_source_id"],
        "publication_date": raw["_publication_date"],
        "publication_date_basis": raw["_publication_date_basis"],
        "period_end": raw.Month,
        "geography_level": level,
        "geography_system": pd.NA,
        "org_code": raw.Org_ONS_Code,
        "org_name": raw.Org_Name,
        "age_band": raw.Age,
        "age_band_source": "published",
        "eligible": to_num(raw.Eligible),
        "screened": to_num(raw.Screened),
        "coverage_pct_published": to_num(raw["Coverage(%)"]),
        "screens_to_target": to_num(raw[screens_col]) if screens_col else pd.NA,
        "quarter": raw.Quarter,
    })
    d["comparable"] = d.quarter != NOT_COMPARABLE_QUARTER
    d["comparability_note"] = np.where(d.comparable, "", NHAIS_NOTE)
    return finish(d)

mi = pd.concat([conform_mi("mi_quarterly_la", "Local Authority"),
                conform_mi("mi_quarterly_subicb", "Sub-ICB")], ignore_index=True)

print(f"mi: {len(mi):,} rows, {mi.quarter.nunique()} quarters, "
      f"{(~mi.comparable).sum():,} rows flagged not comparable")
mi.groupby(["geography_level", "comparable"]).size()

mi: 6,714 rows, 14 quarters, 212 rows flagged not comparable


geography_level  comparable
Local Authority  True          3958
Sub-ICB          False          212
                 True          2544
dtype: int64

## 4. Conform the GOV.UK regional tables

Bronze landed these as raw grids because locating the table was a judgement call. Here is the
judgement: four title rows, a blank, the header on row 5, data from row 6.

Three things get fixed. Coverage arrives as a proportion — `0.7124` where every other source says
`71.24` — and is scaled. The age band comes from the sheet name rather than any column. And the
reference date is parsed out of the title row, which lets the publisher's own statement be checked
against the date I typed into the Bronze manifest.

The regions here are the seven NHS England regions plus an England total. The annual file uses nine
ONS statistical regions. Different carve-ups of the same country, so they are tagged as different
systems and can never be joined by accident.

In [9]:
HEADER_ROW, FIRST_DATA_ROW = 5, 6
PERIOD_RE = re.compile(r"For period up to\s+(.+)$", re.I)

def conform_govuk(source_id_prefix):
    frames = []
    for path in sorted(TABLES_DIR.glob(f"{source_id_prefix}__*.parquet")):
        raw = pd.read_parquet(path)
        grid = raw[[c for c in raw.columns if not c.startswith("_")]]

        # the publisher states the reference date in a title row
        title = str(grid.iloc[2, 0])
        m = PERIOD_RE.search(title)
        period_end = pd.to_datetime(m.group(1).strip(), dayfirst=True).strftime("%Y-%m-%d")

        band = "25-49" if "25to49" in raw["_member"].iloc[0] else "50-64"
        body = grid.iloc[FIRST_DATA_ROW:].copy()
        body = body[body.iloc[:, 0].notna()]

        d = pd.DataFrame({
            "series": raw["_series"].iloc[0],
            "source_id": raw["_source_id"].iloc[0],
            "publication_date": raw["_publication_date"].iloc[0],
            "publication_date_basis": raw["_publication_date_basis"].iloc[0],
            "period_end": period_end,
            "org_name": body.iloc[:, 0].str.strip().values,
            "age_band": band,
            "age_band_source": "published",
            "eligible": to_num(body.iloc[:, 1]).values,
            "screened": to_num(body.iloc[:, 2]).values,
            # proportion in this source, percentage everywhere else
            "coverage_pct_published": (to_num(body.iloc[:, 3]) * 100).values,
            "screens_to_target": to_num(body.iloc[:, 5]).values,
            "comparable": True,
            "comparability_note": "",
        })
        is_england = d.org_name.str.casefold() == "england"
        d["geography_level"]  = np.where(is_england, "National", "Region")
        d["geography_system"] = np.where(is_england, pd.NA, "NHSE")
        frames.append(finish(d))
    return pd.concat(frames, ignore_index=True)

govuk = pd.concat([conform_govuk(s) for s in
                   ["govuk_mar_2023", "govuk_jun_2023", "govuk_sep_2023",
                    "govuk_dec_2023", "govuk_mar_2024"]], ignore_index=True)

print(f"govuk: {len(govuk):,} rows, periods {sorted(govuk.period_end.unique())}")
print(f"NHS regions: {sorted(govuk[govuk.geography_level=='Region'].org_name.unique())}")

govuk: 80 rows, periods ['2023-03-31', '2023-06-30', '2023-09-30', '2023-12-31', '2024-03-31']
NHS regions: ['East Of England', 'London', 'Midlands', 'North East And Yorkshire', 'North West', 'South East', 'South West']


## 5. Union, derive, and compute

Three additions after the union.

Coverage is recomputed from screened over eligible for every row and kept beside the published
figure. Nothing downstream reads the published column — it exists so the two can be checked against
each other.

The 25-64 band is derived wherever a source publishes only the two component bands. Summing them is
valid because they are disjoint counts, but it is my arithmetic, so those rows are marked `derived`
and must never be presented as published.

Lag is publication date minus period end, in days.

In [11]:
coverage = pd.concat([annual, mi, govuk], ignore_index=True)

# Recompute rather than trust. The published column stays for comparison only.
coverage["coverage_pct_computed"] = 100 * coverage.screened / coverage.eligible

KEY = ["series", "source_id", "publication_date", "publication_date_basis",
       "period_end", "geography_level", "geography_system", "org_code", "org_name",
       "quarter", "comparable", "comparability_note"]

# Derive 25-64 only where a source publishes the two parts but not the whole.
parts = coverage[coverage.age_band.isin(["25-49", "50-64"])]
have  = coverage[coverage.age_band == "25-64"][["source_id", "period_end", "org_name"]].drop_duplicates()

grouped = (parts.groupby(KEY, dropna=False)
                .agg(eligible=("eligible", "sum"),
                     screened=("screened", "sum"),
                     screens_to_target=("screens_to_target", "sum"),
                     n_bands=("age_band", "nunique"))
                .reset_index())
grouped = grouped[grouped.n_bands == 2].drop(columns="n_bands")

derived = grouped.merge(have, on=["source_id", "period_end", "org_name"],
                        how="left", indicator=True)
derived = derived[derived._merge == "left_only"].drop(columns="_merge")
derived["age_band"] = "25-64"
derived["age_band_source"] = "derived"
derived["coverage_pct_published"] = pd.NA
derived["coverage_pct_computed"] = 100 * derived.screened / derived.eligible

coverage = pd.concat([coverage, finish(derived)], ignore_index=True)

coverage["period_end"] = pd.to_datetime(coverage.period_end)
coverage["publication_date"] = pd.to_datetime(coverage.publication_date)
coverage["lag_days"] = (coverage.publication_date - coverage.period_end).dt.days

print(f"{len(coverage):,} rows")
print(coverage.groupby(["series", "age_band_source"]).size().to_string())

11,181 rows
series           age_band_source
annual           published           990
govuk_quarterly  derived              40
                 published            80
mi_quarterly     derived            3357
                 published          6714


## 6. Publications and exclusions

Two small tables beside the big one.

`publications` is one row per publication event and feeds the timeline chart. It includes the
rejected sources, because a file I chose not to read was still a publication that happened.

`exclusions` records what Silver dropped and why. A layer that makes decisions should record them as
data, not in a comment nobody reads.

In [13]:
publications = manifest[["series", "source_id", "publication_title", "publisher",
                         "source_page_url", "publication_date", "publication_date_basis",
                         "reference_period_end", "geography_levels", "status", "note"]].copy()
publications["publication_date"] = pd.to_datetime(publications.publication_date)
publications["reference_period_end"] = pd.to_datetime(publications.reference_period_end)
publications["lag_days"] = (publications.publication_date
                            - publications.reference_period_end).dt.days
publications = publications.sort_values("reference_period_end")

exclusions = pd.DataFrame([
    dict(scope="bronze source", ref=r.source_id, reason=r.note)
    for r in manifest[manifest.status == "rejected"].itertuples()
] + [
    dict(scope="bronze source", ref="annual_kc53_coverage",
         reason="Fine age bands (<20, 20-24, 25-29 ...) do not conform to the PHOF 25-49 / 50-64 "
                "vocabulary, and the national trend it would provide is already in annual_phof. "
                "Landed in Bronze, not carried into the conformed table."),
])

print(f"publications: {len(publications)} rows ({(publications.status=='used').sum()} used)")
print(f"exclusions: {len(exclusions)} rows")
publications[["series", "source_id", "reference_period_end", "publication_date", "lag_days", "status"]]

publications: 13 rows (9 used)
exclusions: 5 rows


,series,source_id,reference_period_end,publication_date,lag_days,status
7,govuk_quarterly,govuk_mar_2023,2023-03-31,2023-10-31,214,used
8,govuk_quarterly,govuk_jun_2023,2023-06-30,2023-10-31,123,used
9,govuk_quarterly,govuk_sep_2023,2023-09-30,2023-10-31,31,used
10,govuk_quarterly,govuk_dec_2023,2023-12-31,2024-04-26,117,used
0,annual,annual_phof,2024-03-31,2024-11-28,242,used
1,annual,annual_kc53_coverage,2024-03-31,2024-11-28,242,used
2,annual,annual_workbook,2024-03-31,2024-11-28,242,rejected
3,mi_quarterly,mi_quarterly_la,2024-03-31,2025-01-07,282,used
11,govuk_quarterly,govuk_mar_2024,2024-03-31,2025-07-16,472,used
12,govuk_quarterly,govuk_mar_2024_subicb_gp,2024-03-31,2025-07-16,472,rejected


## 7. Checks

The reconciliation from Bronze is repeated here, on the conformed table, because a conforming step
is exactly where a correct number quietly becomes a wrong one.

Two of these are not assertions but findings. The annual file has 151 local authorities at
31 March 2024; the quarterly file has 153 at the same date. Silver's job is to surface that, not to
pick one silently.

In [15]:
checks = []
def check(name, ok, detail=""):
    checks.append(dict(check=name, passed=bool(ok), detail=str(detail)))

MAR_2024 = pd.Timestamp("2024-03-31")

# 1. the headline still reconciles after conforming
la_annual = coverage[(coverage.series == "annual") & (coverage.period_end == MAR_2024)
                     & (coverage.geography_level == "Local Authority")
                     & (coverage.age_band == "25-64")]
derived_pct = 100 * la_annual.screened.sum() / la_annual.eligible.sum()
print(f"England from {len(la_annual)} LA rows: {derived_pct:.4f}%")
check("headline reconciles to 68.8%", round(derived_pct, 1) == 68.8, f"{derived_pct:.4f}")

# 2. published and recomputed coverage agree wherever both exist
both = coverage[coverage.coverage_pct_published.notna() & coverage.coverage_pct_computed.notna()]
worst = (both.coverage_pct_published - both.coverage_pct_computed).abs().max()
print(f"largest published vs recomputed gap: {worst:.6f} pp over {len(both):,} rows")
check("published and recomputed coverage agree", worst < 0.01, f"max {worst:.6f} pp")

# 3. the date parsed from each GOV.UK title row matches the Bronze manifest
gk = (coverage[coverage.series == "govuk_quarterly"][["source_id", "period_end"]]
      .drop_duplicates()
      .merge(manifest[["source_id", "reference_period_end"]], on="source_id"))
gk["agrees"] = gk.period_end == pd.to_datetime(gk.reference_period_end)
print(gk.to_string(index=False))
check("GOV.UK title-row dates match the manifest", gk.agrees.all())

# 4. the two region systems stay apart
ons  = set(coverage[(coverage.geography_system == "ONS")].org_name)
nhse = set(coverage[(coverage.geography_system == "NHSE")].org_name)
print(f"ONS regions {len(ons)}, NHSE regions {len(nhse)}, shared names {len(ons & nhse)}")
check("9 ONS statistical regions", len(ons) == 9, len(ons))
check("7 NHS England regions", len(nhse) == 7, len(nhse))

# 5. comparability flag lands only on the NHAIS closure quarter
flagged = set(coverage[~coverage.comparable.fillna(True)].quarter.dropna())
check("only 2024-25 Q1 flagged not comparable", flagged == {NOT_COMPARABLE_QUARTER}, flagged)

# 6. derived rows never claim to be published
check("no derived row carries a published coverage",
      coverage[coverage.age_band_source == "derived"].coverage_pct_published.isna().all())

# 7. nothing essential is null
for c in ["series", "source_id", "period_end", "geography_level", "age_band", "eligible", "screened"]:
    check(f"no nulls in {c}", coverage[c].notna().all(), int(coverage[c].isna().sum()))

results = pd.DataFrame(checks)
print(f"\n{results.passed.sum()}/{len(results)} checks passed")
if not results.passed.all():
    print(results[~results.passed].to_string(index=False))

# --- findings, not assertions ---
la_mi = coverage[(coverage.series == "mi_quarterly") & (coverage.period_end == MAR_2024)
                 & (coverage.geography_level == "Local Authority") & (coverage.age_band == "25-64")]
print(f"\nFINDING  local authorities at 31 Mar 2024: annual {la_annual.org_name.nunique()}, "
      f"quarterly {la_mi.org_name.nunique()}")
only_mi = sorted(set(la_mi.org_name) - set(la_annual.org_name))
only_an = sorted(set(la_annual.org_name) - set(la_mi.org_name))
print(f"         in quarterly only: {only_mi}")
print(f"         in annual only:    {only_an}")

both_src = (la_annual[["org_name", "eligible", "screened"]].merge(
            la_mi[["org_name", "eligible", "screened"]], on="org_name", suffixes=("_ann", "_mi")))
both_src["pp_gap"] = (100 * both_src.screened_ann / both_src.eligible_ann
                      - 100 * both_src.screened_mi / both_src.eligible_mi)
print(f"\nFINDING  same LA, same date, two publications: largest coverage gap "
      f"{both_src.pp_gap.abs().max():.4f} pp across {len(both_src)} authorities")

assert results.passed.all(), "A Silver check failed."

England from 151 LA rows: 68.7988%
largest published vs recomputed gap: 0.000000 pp over 7,784 rows
     source_id period_end reference_period_end  agrees
govuk_mar_2023 2023-03-31           2023-03-31    True
govuk_jun_2023 2023-06-30           2023-06-30    True
govuk_sep_2023 2023-09-30           2023-09-30    True
govuk_dec_2023 2023-12-31           2023-12-31    True
govuk_mar_2024 2024-03-31           2024-03-31    True
ONS regions 9, NHSE regions 7, shared names 4

14/14 checks passed

FINDING  local authorities at 31 Mar 2024: annual 151, quarterly 153
         in quarterly only: ['City of London', 'Isles of Scilly']
         in annual only:    []

FINDING  same LA, same date, two publications: largest coverage gap 0.4330 pp across 151 authorities


## 8. Save and summarise

In [17]:
coverage.to_parquet(SILVER_DIR / "coverage.parquet", index=False)
publications.to_parquet(SILVER_DIR / "publications.parquet", index=False)
publications.to_csv(SILVER_DIR / "publications.csv", index=False)
exclusions.to_csv(SILVER_DIR / "exclusions.csv", index=False)

print(f"coverage      {len(coverage):>7,} rows -> silver/coverage.parquet")
print(f"publications  {len(publications):>7,} rows -> silver/publications.parquet")
print(f"exclusions    {len(exclusions):>7,} rows -> silver/exclusions.csv")
print()
print(coverage.groupby(["series", "geography_level"])
              .agg(rows=("org_name", "size"),
                   orgs=("org_name", "nunique"),
                   first=("period_end", "min"),
                   last=("period_end", "max")).to_string())

coverage       11,181 rows -> silver/coverage.parquet
publications       13 rows -> silver/publications.parquet
exclusions          5 rows -> silver/exclusions.csv

                                 rows  orgs      first       last
series          geography_level                                  
annual          Local Authority   903   152 2023-03-31 2024-03-31
                National           33     1 2014-03-31 2024-03-31
                Region             54     9 2023-03-31 2024-03-31
govuk_quarterly National           15     1 2023-03-31 2024-03-31
                Region            105     7 2023-03-31 2024-03-31
mi_quarterly    Local Authority  5937   155 2021-03-31 2024-03-31
                Sub-ICB          4134   319 2021-06-30 2024-06-30
